<a href="https://colab.research.google.com/github/ElionLAB/OOP_2026_Practice/blob/main/ch_04/src/part_1/Lec_4_part_1_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Environment Setup — Auto-detect Google Colab / Local
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    pass  # This notebook does not require additional packages.
else:
    print('Local environment: Make sure `conda activate oop_practice` is active.')

# Lecture 4 — Part 1: Inheritance, `super()`, MRO, and Polymorphism

**Konkuk University OOP (Python Object-Oriented Programming) — Spring 2026**

---

## Learning Objectives

1. Use `super()` correctly to initialize parent classes
2. Understand Method Resolution Order (MRO) and the diamond problem
3. Write cooperative multiple inheritance with `**kwargs`
4. Apply polymorphism and duck typing
5. Design an abstract base class with concrete subclasses (Distance case study)

## How to use this notebook

Each section has a **Full code reference** (shown first as a goal) and then a **TODO** cell where you fill in the blanks marked as `# TODO`. Run the test cell after each exercise to check your solution.

## 1. `super()` in Initialization (Slide 7)

When a subclass defines its own `__init__`, it must still initialize the parent class.
The idiomatic way is to call `super().__init__(...)` and pass the arguments the parent expects.

### Goal

Build a small inheritance chain:
- `Contact` stores `name` and `email`.
- `Friend` extends `Contact` by adding `phone`.
- `Friend.__init__` must call `super().__init__(name, email)` to let the parent do its job.

In [ ]:
# TODO 1: Complete the classes below.

class Contact:
    all_contacts: list["Contact"] = []

    def __init__(self, name: str, email: str) -> None:
        # TODO 1-a: assign self.name and self.email
        self.name = ...
        self.email = ...
        Contact.all_contacts.append(self)

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(name={self.name!r}, email={self.email!r})"


class Friend(Contact):
    def __init__(self, name: str, email: str, phone: str) -> None:
        # TODO 1-b: call the parent initializer using super() so that
        #          self.name and self.email get set by Contact.__init__
        ...
        # TODO 1-c: store the new attribute specific to Friend
        self.phone = ...

In [ ]:
# Quick check
f = Friend("Alice", "alice@konkuk.ac.kr", "010-1234-5678")
assert f.name == "Alice"
assert f.email == "alice@konkuk.ac.kr"
assert f.phone == "010-1234-5678"
assert f in Contact.all_contacts
print("OK:", f)

## 2. MRO + `super()` to the Rescue (Slide 12)

Python resolves method lookup with the **Method Resolution Order (MRO)**, computed by the C3 linearization algorithm.
When you use `super()`, Python walks the MRO — not just a single parent — which lets cooperative multiple inheritance work.

### Goal: the diamond

```
        object
          |
        Contact
        /     \
  Friend     Emailable
        \     /
       EmailableFriend
```

Every class's `__init__` should call `super().__init__(...)` so that **all** branches of the diamond get initialized exactly once.

### Full code reference

```python
class Contact:
    def __init__(self, name: str = "", email: str = "", **kwargs):
        super().__init__(**kwargs)
        self.name = name
        self.email = email


class AddressHolder:
    def __init__(self, street: str = "", city: str = "", **kwargs):
        super().__init__(**kwargs)
        self.street = street
        self.city = city


class Friend(Contact, AddressHolder):
    def __init__(self, phone: str = "", **kwargs):
        super().__init__(**kwargs)
        self.phone = phone
```

Inspecting the MRO:

```python
print([c.__name__ for c in Friend.__mro__])
# ['Friend', 'Contact', 'AddressHolder', 'object']
```

In [ ]:
# TODO 2: Fill in the diamond. Every __init__ must cooperate with super().

class Contact:
    def __init__(self, name: str = "", email: str = "", **kwargs):
        # TODO 2-a: forward remaining kwargs to the next class in the MRO
        ...
        self.name = name
        self.email = email


class AddressHolder:
    def __init__(self, street: str = "", city: str = "", **kwargs):
        # TODO 2-b: forward remaining kwargs
        ...
        self.street = street
        self.city = city


class Friend(Contact, AddressHolder):
    def __init__(self, phone: str = "", **kwargs):
        # TODO 2-c: forward remaining kwargs
        ...
        self.phone = phone


# TODO 2-d: print the MRO of Friend as a list of class names
print(...)

In [ ]:
# Quick check
f = Friend(
    name="Bob",
    email="bob@konkuk.ac.kr",
    street="120 Neungdong-ro",
    city="Seoul",
    phone="010-0000-1111",
)
assert f.name == "Bob"
assert f.email == "bob@konkuk.ac.kr"
assert f.street == "120 Neungdong-ro"
assert f.city == "Seoul"
assert f.phone == "010-0000-1111"
print("OK — every branch of the diamond was initialized.")

## 3. Cooperative `**kwargs` (Slide 14)

**Why `**kwargs`?** In cooperative multiple inheritance, you don't know in advance which class comes next in the MRO or what arguments it needs. Passing `**kwargs` up the chain lets each class pick off the arguments it recognizes and forward the rest.

### Rules

1. Every cooperative `__init__` must accept `**kwargs`.
2. Every cooperative `__init__` must end its argument-consuming work with `super().__init__(**kwargs)`.
3. At the top, `object.__init__` receives an **empty** `kwargs` (otherwise you get a `TypeError`).

In [ ]:
# TODO 3: Build a cooperative hierarchy for a contact-manager app.
#         Each mixin adds one piece of data and forwards the rest.

class BaseContact:
    def __init__(self, name: str = "", **kwargs):
        # TODO 3-a: cooperative call to super()
        ...
        self.name = name


class EmailMixin:
    def __init__(self, email: str = "", **kwargs):
        # TODO 3-b
        ...
        self.email = email


class PhoneMixin:
    def __init__(self, phone: str = "", **kwargs):
        # TODO 3-c
        ...
        self.phone = phone


class FullContact(EmailMixin, PhoneMixin, BaseContact):
    """No __init__ needed — inherits cooperative behavior."""
    pass

In [ ]:
# Quick check
c = FullContact(name="Carol", email="carol@konkuk.ac.kr", phone="010-9999-8888")
assert (c.name, c.email, c.phone) == ("Carol", "carol@konkuk.ac.kr", "010-9999-8888")
print("MRO:", [cls.__name__ for cls in FullContact.__mro__])
print("OK:", vars(c))

## 4. Polymorphism and Duck Typing (Slide 16)

**Polymorphism**: different classes respond to the *same* method name in their own way.
**Duck typing**: "If it walks like a duck and quacks like a duck, it is a duck." Python does not check types — it just calls the method.

### Goal — Full code

You will build a tiny audio library where each file type knows how to `play()` itself. The `play_all(files)` function does not care about the concrete class; it only requires that `play()` exists.

### Full code reference

```python
class AudioFile:
    ext: str  # subclasses must define this

    def __init__(self, filename: str) -> None:
        if not filename.endswith("." + self.ext):
            raise ValueError(f"Invalid file format for {self.__class__.__name__}")
        self.filename = filename

    def play(self) -> None:
        raise NotImplementedError


class MP3File(AudioFile):
    ext = "mp3"
    def play(self) -> None:
        print(f"playing {self.filename} as mp3")


class WavFile(AudioFile):
    ext = "wav"
    def play(self) -> None:
        print(f"playing {self.filename} as wav")


class OggFile(AudioFile):
    ext = "ogg"
    def play(self) -> None:
        print(f"playing {self.filename} as ogg")


# Duck typing — this class is NOT an AudioFile, but it still works with play_all
class FlacPlayer:
    def __init__(self, filename: str) -> None:
        self.filename = filename
    def play(self) -> None:
        print(f"playing {self.filename} as flac (duck-typed!)")


def play_all(files):
    for f in files:
        f.play()
```

In [ ]:
# TODO 4: Reproduce the audio library from the full code reference.
#         Do NOT copy verbatim — type it yourself so it sticks.

class AudioFile:
    ext: str  # subclasses must override

    def __init__(self, filename: str) -> None:
        # TODO 4-a: raise ValueError if filename does not end with "." + self.ext
        ...
        self.filename = filename

    def play(self) -> None:
        # TODO 4-b: subclasses must implement this; raise NotImplementedError
        ...


class MP3File(AudioFile):
    ext = "mp3"
    def play(self) -> None:
        # TODO 4-c
        ...


class WavFile(AudioFile):
    ext = "wav"
    def play(self) -> None:
        # TODO 4-d
        ...


class OggFile(AudioFile):
    ext = "ogg"
    def play(self) -> None:
        # TODO 4-e
        ...


class FlacPlayer:
    """Duck-typed: no inheritance from AudioFile, yet play_all accepts it."""
    def __init__(self, filename: str) -> None:
        self.filename = filename
    def play(self) -> None:
        # TODO 4-f
        ...


def play_all(files):
    # TODO 4-g: call play() on each element, relying on duck typing
    ...

In [ ]:
# Quick check
play_all([
    MP3File("song.mp3"),
    WavFile("voice.wav"),
    OggFile("podcast.ogg"),
    FlacPlayer("studio.flac"),
])

# Invalid extension must raise
try:
    MP3File("not_an_mp3.wav")
except ValueError as e:
    print("OK, caught:", e)
else:
    raise AssertionError("MP3File should have rejected a .wav filename")

## 5. Concrete Distances (Slide 25)

A common OOP case study: a **k-Nearest-Neighbors** classifier needs to measure the distance between two samples. There are many distance formulas — Euclidean, Manhattan, Chebyshev — but they all share the same interface: `distance(s1, s2) -> float`.

We model this with:
- an **abstract base class** `Distance` that declares the interface, and
- **concrete subclasses** that each provide a formula.

This is textbook polymorphism: the kNN classifier only depends on the *interface*, not the concrete class.

### Full code reference

```python
from abc import ABC, abstractmethod
from math import hypot

class Sample:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y


class Distance(ABC):
    @abstractmethod
    def distance(self, s1: Sample, s2: Sample) -> float:
        ...


class EuclideanDistance(Distance):
    def distance(self, s1: Sample, s2: Sample) -> float:
        return hypot(s1.x - s2.x, s1.y - s2.y)


class ManhattanDistance(Distance):
    def distance(self, s1: Sample, s2: Sample) -> float:
        return abs(s1.x - s2.x) + abs(s1.y - s2.y)


class ChebyshevDistance(Distance):
    def distance(self, s1: Sample, s2: Sample) -> float:
        return max(abs(s1.x - s2.x), abs(s1.y - s2.y))
```

In [ ]:
# TODO 5: Implement the concrete distance classes.

from abc import ABC, abstractmethod
from math import hypot


class Sample:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y


class Distance(ABC):
    @abstractmethod
    def distance(self, s1: Sample, s2: Sample) -> float:
        ...


class EuclideanDistance(Distance):
    def distance(self, s1: Sample, s2: Sample) -> float:
        # TODO 5-a: return sqrt((dx)^2 + (dy)^2). Hint: use math.hypot
        ...


class ManhattanDistance(Distance):
    def distance(self, s1: Sample, s2: Sample) -> float:
        # TODO 5-b: return |dx| + |dy|
        ...


class ChebyshevDistance(Distance):
    def distance(self, s1: Sample, s2: Sample) -> float:
        # TODO 5-c: return max(|dx|, |dy|)
        ...

In [ ]:
# Quick check
a, b = Sample(0.0, 0.0), Sample(3.0, 4.0)
assert abs(EuclideanDistance().distance(a, b) - 5.0) < 1e-9
assert ManhattanDistance().distance(a, b) == 7.0
assert ChebyshevDistance().distance(a, b) == 4.0
print("All three distance formulas pass.")

# Try instantiating the abstract class — this MUST fail:
try:
    Distance()
except TypeError as e:
    print("OK, abstract class cannot be instantiated:", e)

## 6. Exercises (Slide 27)

Three short exercises to consolidate everything above. Each has a `# TODO` to fill and a test cell.

### Exercise 1 — `super()` in a three-level chain

Define `Animal` → `Dog` → `Puppy`.

- `Animal.__init__(self, species)` sets `self.species`.
- `Dog.__init__(self, name)` calls the parent (passing `species="dog"`) and sets `self.name`.
- `Puppy.__init__(self, name, age_months)` calls the parent and sets `self.age_months`.

Every subclass must use `super().__init__(...)` — do **not** write `Animal.__init__(self, ...)` directly.

In [ ]:
# TODO Exercise 1

class Animal:
    def __init__(self, species: str) -> None:
        # TODO E1-a
        ...


class Dog(Animal):
    def __init__(self, name: str) -> None:
        # TODO E1-b: call super() with species="dog"
        ...
        # TODO E1-c: store name
        ...


class Puppy(Dog):
    def __init__(self, name: str, age_months: int) -> None:
        # TODO E1-d
        ...
        # TODO E1-e
        ...

In [ ]:
# Quick check
p = Puppy("Coco", 3)
assert p.species == "dog"
assert p.name == "Coco"
assert p.age_months == 3
print("Exercise 1 OK:", vars(p))

### Exercise 2 — Cooperative mixins with `**kwargs`

Design a `User` that gains features via mixins:

- `TimestampedMixin` adds `self.created_at` (use `datetime.datetime.now()`).
- `TaggedMixin` adds `self.tags` from a `tags` keyword argument (default `[]`).
- `BaseUser` stores `self.username`.

Define `User(TimestampedMixin, TaggedMixin, BaseUser)`. Every `__init__` must accept and forward `**kwargs`.

In [ ]:
# TODO Exercise 2
import datetime


class BaseUser:
    def __init__(self, username: str = "", **kwargs):
        # TODO E2-a
        ...
        self.username = username


class TimestampedMixin:
    def __init__(self, **kwargs):
        # TODO E2-b
        ...
        self.created_at = datetime.datetime.now()


class TaggedMixin:
    def __init__(self, tags=None, **kwargs):
        # TODO E2-c
        ...
        self.tags = list(tags) if tags else []


class User(TimestampedMixin, TaggedMixin, BaseUser):
    pass

In [ ]:
# Quick check
u = User(username="dayoung", tags=["student", "oop"])
assert u.username == "dayoung"
assert u.tags == ["student", "oop"]
assert isinstance(u.created_at, datetime.datetime)
print("Exercise 2 OK:", u.username, u.tags, u.created_at)
print("MRO:", [c.__name__ for c in User.__mro__])

### Exercise 3 — One more concrete distance

Extend the Distance hierarchy from Section 5 with a **Minkowski distance** of order `p`:

$$ d(s_1, s_2) = \left(|x_1 - x_2|^p + |y_1 - y_2|^p\right)^{1/p} $$

- For `p=1` it equals the Manhattan distance.
- For `p=2` it equals the Euclidean distance.
- `p` should be a constructor argument.

Your class must inherit from `Distance` (reuse the abstract base class from Section 5).

In [ ]:
# TODO Exercise 3

class MinkowskiDistance(Distance):
    def __init__(self, p: float) -> None:
        # TODO E3-a: store p
        ...

    def distance(self, s1: Sample, s2: Sample) -> float:
        # TODO E3-b: implement the Minkowski formula
        ...

In [ ]:
# Quick check
a, b = Sample(0.0, 0.0), Sample(3.0, 4.0)

assert abs(MinkowskiDistance(p=1).distance(a, b) - 7.0) < 1e-9   # Manhattan
assert abs(MinkowskiDistance(p=2).distance(a, b) - 5.0) < 1e-9   # Euclidean
assert abs(MinkowskiDistance(p=3).distance(a, b) - (3**3 + 4**3) ** (1/3)) < 1e-9
print("Exercise 3 OK.")

---

## Summary

- `super()` delegates to the **next class in the MRO**, not to a fixed parent.
- Cooperative multiple inheritance needs `**kwargs` + `super().__init__(**kwargs)` in every class.
- Duck typing means: *if the method exists, Python calls it* — no type check required.
- Abstract base classes (`ABC` + `@abstractmethod`) let you pin an interface while leaving the implementation to subclasses.

**Next:** Part 2 will cover abstract base classes in more depth, `isinstance` vs duck typing, and when to prefer composition over inheritance.